In [ ]:

"""
Dog Robot — Pure NN + PPO  (ΔΔθ actions — acceleration-level control)
=================================================================================
The actor outputs ΔΔθ_i each step — a small change to a PERSISTENT DELTA BUFFER.
The delta buffer is then added to the current joint target.

  action  →  delta  ←  delta + action * MAX_DDELTA  (clipped to ±DELTA_LIMIT)
  ctrl    ←  ctrl   +  delta                         (clipped to joint limits)

This adds joint-space inertia: the network cannot instantly reverse direction.
Any change in joint velithout any explicit frequency constraint.
ocity must be accumulated over multiple steps, enforcing
structural smoothness w
State  (24): jpos(8) + jvel(8) + delta(8)
Action  (8): ΔΔθ_i  (scaled by MAX_DDELTA, added to current delta buffer)
Actor arch:  24 → HIDDEN_DIM → 8  (single hidden layer, ReLU, tanh output)
             Small enough to run inference on an ESP32 at control frequency.
Reward:      -VX_TRACK_W*(vx - TARGET_VX)²  + alive - 0.1|vy|
             - ACTION_REG_W||ΔΔθ||² - DELTA_REG_W||delta||²
             + JVEL_W * jvel_mag + JEXC_W * excursion
             - ALIGN_W*(roll²+pitch²) - YAW_W*Δyaw²

Target speed: TARGET_VX = 1m / 8s = 0.125 m/s
The tracking reward is zero at exactly TARGET_VX and falls off as (vx - TARGET_VX)².
A dead-zone of ±VX_TOL suppresses the penalty for near-target speeds so the
policy is not penalised for small natural fluctuations around the setpoint.
"""

# ── Hyper-parameters ───────────────────────────────────────────
N_EP         = 10000
ALIGN_W      = 0.00
YAW_W        = 0.20
ALIVE_BONUS  = 0.05

MAX_DDELTA   = 0.02          # rad per step — acceleration limit (ΔΔθ scale)
DELTA_LIMIT  = 0.05          # rad per step — cap on delta buffer (same as old MAX_DELTA)

ACTION_REG_W = 0.0001        # penalty on ΔΔθ magnitude (keeps accelerations small)
DELTA_REG_W  = 0.0002        # penalty on delta magnitude (prevents drift)
JVEL_W       = 0.05          # reward active joint movement
JEXC_W       = 0.6          # reward joint excursion from neutral

# ── Velocity tracking ──────────────────────────────────────────
TARGET_VX   = 1.0 / 4.0      # 0.125 m/s  (2 m in 8 s)
VX_TRACK_W  = 4.0            # weight on (vx - TARGET_VX)²; 0.25 m/s off → −0.25 reward
VX_TOL      = 0.02           # dead-zone: no penalty if |vx - TARGET_VX| < VX_TOL

import os, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import mujoco
import mediapy as media
import matplotlib.pyplot as plt
from tqdm import trange

os.makedirs("movies",       exist_ok=True)
os.makedirs("models",       exist_ok=True)
os.makedirs("plots",        exist_ok=True)
os.makedirs("trajectories", exist_ok=True)

# ── Robot geometry ─────────────────────────────────────────────
HIP_INIT  =  0.90
KNEE_INIT = -1.40
DELTA_HIP  = 0.50
DELTA_KNEE = 0.30
HIP_LO,  HIP_HI  = HIP_INIT - DELTA_HIP,  HIP_INIT + DELTA_HIP
KNEE_LO, KNEE_HI = KNEE_INIT - DELTA_KNEE, KNEE_INIT + DELTA_KNEE
OFFSETS   = np.array([HIP_INIT, KNEE_INIT] * 4, dtype=np.float32)
JOINT_LO  = np.array([HIP_LO, KNEE_LO] * 4, dtype=np.float32)
JOINT_HI  = np.array([HIP_HI, KNEE_HI] * 4, dtype=np.float32)

SUBSTEPS    = 5
SETTLE_TIME = 0.3
EPISODE_DUR = 8.0

# ── PPO hyper-parameters ──────────────────────────────────────
LR           = 3e-4
GAMMA        = 0.99
GAE_LAMBDA   = 0.95
CLIP_EPS     = 0.2
ENTROPY_COEF = 0.005
VF_COEF      = 0.5
MAX_GRAD     = 0.5

N_STEPS_PER_UPDATE = 4096
PPO_EPOCHS         = 10
MINIBATCH_SIZE     = 256

LOG_STD_INIT = 0.5
LOG_STD_MIN  = -2.0
LOG_STD_MAX  = 1.0

RENDER_EVERY = 200

LOAD_MODEL = ""  # e.g. "models/nn_ppo_ddtheta_best.pth"
EVAL_ONLY  = False

# ── Dimensions ─────────────────────────────────────────────────
ACTION_DIM = 8
HIDDEN_DIM = 64              # single hidden layer — fits easily in ESP32 SRAM
#                              W1: 64×24=1536 f32 (6 KB)  W2: 8×64=512 f32 (2 KB)
STATE_DIM  = 24              # jpos(8) + jvel(8) + delta(8)

mjmodel = mujoco.MjModel.from_xml_path("dog.xml")
mjdata  = mujoco.MjData(mjmodel)
CTRL_DT = mjmodel.opt.timestep * SUBSTEPS
print(f"CTRL_DT = {CTRL_DT*1000:.1f} ms  ({1/CTRL_DT:.0f} Hz)")
print(f"State dim = {STATE_DIM}  |  Action dim = {ACTION_DIM}")
print(f"MAX_DDELTA = {MAX_DDELTA} rad/step  |  DELTA_LIMIT = {DELTA_LIMIT} rad/step")


# ══════════════════════════════════════════════════════════════
#  Helpers
# ══════════════════════════════════════════════════════════════
def orientation():
    qw, qx, qy, qz = mjdata.qpos[3:7]
    roll  = math.atan2(2*(qw*qx + qy*qz), 1 - 2*(qx**2 + qy**2))
    pitch = math.asin(float(np.clip(2*(qw*qy - qz*qx), -1, 1)))
    yaw   = math.atan2(2*(qw*qz + qx*qy), 1 - 2*(qy**2 + qz**2))
    return roll, pitch, yaw


def build_state(delta: np.ndarray) -> np.ndarray:
    """24-dim: jpos(8) + jvel(8) + delta(8).
    The delta buffer is part of the state so the policy can see its momentum."""
    return np.concatenate([
        mjdata.qpos[7:15].astype(np.float32),
        mjdata.qvel[6:14].astype(np.float32),
        delta.astype(np.float32),
    ])


def apply_ddelta(ctrl: np.ndarray, delta: np.ndarray, ddelta: np.ndarray):
    """Two-level integration:
        delta_new = clip(delta + ddelta * MAX_DDELTA, -DELTA_LIMIT, +DELTA_LIMIT)
        ctrl_new  = clip(ctrl  + delta_new,            JOINT_LO,    JOINT_HI)
    Returns (ctrl_new, delta_new).
    """
    delta_new = np.clip(delta + ddelta * MAX_DDELTA, -DELTA_LIMIT, DELTA_LIMIT)
    ctrl_new  = np.clip(ctrl  + delta_new,            JOINT_LO,    JOINT_HI)
    return ctrl_new, delta_new


def is_done():
    _, p, _ = orientation()
    return mjdata.qpos[2] < 0.03 or abs(p) > math.radians(45)


def reset_sim():
    """Reset simulation; returns (ctrl, delta) both zeroed to neutral pose."""
    ip = np.array([HIP_INIT, KNEE_INIT] * 4, dtype=np.float32)
    mujoco.mj_resetData(mjmodel, mjdata)
    mjdata.qpos[7:15] = ip
    mjdata.qpos[2]    = 0.10
    mujoco.mj_forward(mjmodel, mjdata)
    for _ in range(int(SETTLE_TIME / mjmodel.opt.timestep)):
        mjdata.ctrl[:] = ip
        mujoco.mj_step(mjmodel, mjdata)
    delta = np.zeros(ACTION_DIM, dtype=np.float32)   # delta buffer starts at rest
    return ip.copy(), delta


# ══════════════════════════════════════════════════════════════
#  Actor-Critic
#  Actor:  STATE_DIM → HIDDEN_DIM → ACTION_DIM  (1 hidden layer, ReLU)
#  Critic: STATE_DIM → 256 → 256 → 1            (2 hidden layers, can be larger)
# ══════════════════════════════════════════════════════════════
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()

        # ── Actor: deliberately small for ESP32 deployment ────
        self.pi = nn.Sequential(
            nn.Linear(STATE_DIM,  HIDDEN_DIM), nn.ReLU(),
            nn.Linear(HIDDEN_DIM, ACTION_DIM),
        )
        nn.init.orthogonal_(self.pi[-1].weight, gain=0.01)
        nn.init.zeros_(self.pi[-1].bias)

        self.log_std = nn.Parameter(
            torch.full((ACTION_DIM,), LOG_STD_INIT)
        )

        self.vf = nn.Sequential(
            nn.Linear(STATE_DIM, 256), nn.ReLU(),
            nn.Linear(256, 256),       nn.ReLU(),
            nn.Linear(256, 1),
        )
        nn.init.orthogonal_(self.vf[-1].weight, gain=1.0)
        nn.init.zeros_(self.vf[-1].bias)

    def forward_pi(self, s):
        mean    = self.pi(s)
        log_std = self.log_std.clamp(LOG_STD_MIN, LOG_STD_MAX)
        return mean, log_std

    def forward_vf(self, s):
        return self.vf(s).squeeze(-1)

    def get_action(self, s, deterministic=False):
        mean, log_std = self.forward_pi(s)
        std = log_std.exp()
        if deterministic:
            raw = mean
        else:
            raw = mean + std * torch.randn_like(std)
        action = torch.tanh(raw)

        log_prob = (-0.5 * ((raw - mean) / (std + 1e-8))**2
                    - log_std - 0.5 * math.log(2 * math.pi))
        log_prob -= torch.log(1 - action.pow(2) + 1e-6)
        log_prob  = log_prob.sum(-1)

        value = self.forward_vf(s)
        return action, log_prob, value

    def evaluate_actions(self, s, actions_tanh):
        mean, log_std = self.forward_pi(s)
        std = log_std.exp()
        raw = torch.atanh(actions_tanh.clamp(-0.999, 0.999))

        log_prob = (-0.5 * ((raw - mean) / (std + 1e-8))**2
                    - log_std - 0.5 * math.log(2 * math.pi))
        log_prob -= torch.log(1 - actions_tanh.pow(2) + 1e-6)
        log_prob  = log_prob.sum(-1)

        entropy = (log_std + 0.5 * math.log(2 * math.pi * math.e)).sum(-1)
        value   = self.forward_vf(s)
        return log_prob, entropy, value


ac  = ActorCritic().to(device)
opt = torch.optim.Adam(ac.parameters(), lr=LR, eps=1e-5)


# ══════════════════════════════════════════════════════════════
#  Checkpoint loading
# ══════════════════════════════════════════════════════════════
if LOAD_MODEL:
    print(f"Loading checkpoint: {LOAD_MODEL}")
    ck = torch.load(LOAD_MODEL, map_location=device)
    ac.load_state_dict(ck.get("best_w") or ck["ac"])
    if "opt" in ck:
        opt.load_state_dict(ck["opt"])
    best_dist_loaded = ck.get("best_dist", -np.inf)
    ep_loaded        = ck.get("ep", 0)
    rl   = ck.get("rl",   [])
    dl   = ck.get("dl",   [])
    cll  = ck.get("cll",  [])
    all_ = ck.get("all_", [])
    print(f"  Resumed ep={ep_loaded}  best_dist={best_dist_loaded:.3f}m")
else:
    best_dist_loaded = -np.inf
    ep_loaded        = 0
    rl, dl, cll, all_ = [], [], [], []


# ══════════════════════════════════════════════════════════════
#  PPO buffer
# ══════════════════════════════════════════════════════════════
class PPOBuffer:
    def __init__(self):
        self.clear()

    def clear(self):
        self.states    = []
        self.actions   = []
        self.log_probs = []
        self.rewards   = []
        self.values    = []
        self.dones     = []

    def push(self, s, a, lp, r, v, d):
        self.states.append(s)
        self.actions.append(a)
        self.log_probs.append(lp)
        self.rewards.append(r)
        self.values.append(v)
        self.dones.append(d)

    def __len__(self):
        return len(self.states)

    def compute_gae(self, last_value):
        T       = len(self.rewards)
        adv     = np.zeros(T, dtype=np.float32)
        ret     = np.zeros(T, dtype=np.float32)
        values  = np.array(self.values + [last_value], dtype=np.float32)
        rewards = np.array(self.rewards, dtype=np.float32)
        dones   = np.array(self.dones,   dtype=np.float32)
        gae = 0.0
        for t in reversed(range(T)):
            delta  = rewards[t] + GAMMA * values[t+1] * (1-dones[t]) - values[t]
            gae    = delta + GAMMA * GAE_LAMBDA * (1-dones[t]) * gae
            adv[t] = gae
            ret[t] = adv[t] + values[t]
        return ret, adv

    def get_tensors(self, last_value):
        ret, adv = self.compute_gae(last_value)
        S   = torch.FloatTensor(np.array(self.states)).to(device)
        A   = torch.FloatTensor(np.array(self.actions)).to(device)
        LP  = torch.FloatTensor(np.array(self.log_probs)).to(device)
        R   = torch.FloatTensor(ret).to(device)
        Adv = torch.FloatTensor(adv).to(device)
        Adv = (Adv - Adv.mean()) / (Adv.std() + 1e-8)
        return S, A, LP, R, Adv


ppo_buf = PPOBuffer()


# ══════════════════════════════════════════════════════════════
#  PPO update
# ══════════════════════════════════════════════════════════════
def update_ppo(last_value):
    S, A, LP_old, R, Adv = ppo_buf.get_tensors(last_value)
    N = S.shape[0]

    total_pi_loss = 0.0
    total_vf_loss = 0.0
    n_updates     = 0

    for _ in range(PPO_EPOCHS):
        idx = torch.randperm(N, device=device)
        for start in range(0, N, MINIBATCH_SIZE):
            mb     = idx[start : start + MINIBATCH_SIZE]
            s_mb   = S[mb]
            a_mb   = A[mb]
            lp_old = LP_old[mb]
            r_mb   = R[mb]
            adv_mb = Adv[mb]

            lp_new, entropy, v_new = ac.evaluate_actions(s_mb, a_mb)

            ratio   = (lp_new - lp_old).exp()
            surr1   = ratio * adv_mb
            surr2   = ratio.clamp(1-CLIP_EPS, 1+CLIP_EPS) * adv_mb
            pi_loss = -torch.min(surr1, surr2).mean()

            vf_loss = F.mse_loss(v_new, r_mb)

            loss = pi_loss + VF_COEF * vf_loss - ENTROPY_COEF * entropy.mean()

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(ac.parameters(), MAX_GRAD)
            opt.step()

            total_pi_loss += pi_loss.item()
            total_vf_loss += vf_loss.item()
            n_updates     += 1

    ppo_buf.clear()
    return total_vf_loss / max(n_updates, 1), total_pi_loss / max(n_updates, 1)


# ══════════════════════════════════════════════════════════════
#  Training rollout
# ══════════════════════════════════════════════════════════════
def rollout_train():
    ctrl, delta = reset_sim()
    x0          = mjdata.qpos[0]
    _, _, yaw0  = orientation()
    tot_r       = 0.0
    s           = build_state(delta)

    done = False
    for t in range(int(EPISODE_DUR / CTRL_DT)):
        st = torch.FloatTensor(s).unsqueeze(0).to(device)
        with torch.no_grad():
            action, log_prob, value = ac.get_action(st, deterministic=False)
        an = action.squeeze(0).cpu().numpy()   # ΔΔθ in [-1, 1]
        lp = log_prob.item()
        v  = value.item()

        # Two-level integration: ΔΔθ → delta → ctrl
        ctrl, delta = apply_ddelta(ctrl, delta, an)
        mjdata.ctrl[:] = ctrl
        for _ in range(SUBSTEPS):
            mujoco.mj_step(mjmodel, mjdata)

        roll, pitch, yaw = orientation()
        dyaw = (yaw - yaw0 + math.pi) % (2 * math.pi) - math.pi
        vx        = float(mjdata.qvel[0])
        vy        = abs(float(mjdata.qvel[1]))
        jvel_mag  = float(np.mean(np.abs(mjdata.qvel[6:14])))
        jpos      = mjdata.qpos[7:15].astype(np.float32)
        excursion = float(np.mean(np.abs(jpos - OFFSETS)))

        # Velocity tracking: reward is 0 at TARGET_VX, falls as squared error.
        # Dead-zone suppresses noise around the setpoint.
        vx_err  = max(0.0, abs(vx - TARGET_VX) - VX_TOL)
        vx_rew  = -VX_TRACK_W * vx_err ** 2

        r = (vx_rew
             + ALIVE_BONUS
             - 0.10 * vy
             - ACTION_REG_W * float(np.sum(an**2))
             - DELTA_REG_W  * float(np.sum(delta**2))
             + JVEL_W  * jvel_mag
             + JEXC_W  * excursion
             - ALIGN_W * (roll**2 + pitch**2)
             - YAW_W   * dyaw**2)
        tot_r += r
        done = is_done()
        if done:
            r -= 5.0

        ppo_buf.push(s, an, lp, r, v, float(done))
        s = build_state(delta)   # delta is part of the next state
        if done:
            break

    if done:
        last_val = 0.0
    else:
        with torch.no_grad():
            st = torch.FloatTensor(s).unsqueeze(0).to(device)
            last_val = ac.forward_vf(st).item()

    dist = mjdata.qpos[0] - x0
    return tot_r, dist, last_val


# ══════════════════════════════════════════════════════════════
#  Eval rollout  (deterministic)
# ══════════════════════════════════════════════════════════════
def rollout_eval(render=False, fps=60, record_traj=False):
    ctrl, delta = reset_sim()
    x0     = mjdata.qpos[0]
    tot_r  = 0.0
    frames = []
    traj_ctrl  = []
    traj_delta = []    # also record delta buffer evolution
    renderer = None

    if render:
        renderer = mujoco.Renderer(mjmodel, height=1080, width=1920)
        cam = mujoco.MjvCamera()
        mujoco.mjv_defaultFreeCamera(mjmodel, cam)
        cam.distance  = 0.6
        cam.elevation = -20
        fe = max(1, int(1 / (fps * CTRL_DT)))

    s = build_state(delta)

    for t in range(int(EPISODE_DUR / CTRL_DT)):
        with torch.no_grad():
            st = torch.FloatTensor(s).unsqueeze(0).to(device)
            action, _, _ = ac.get_action(st, deterministic=True)
            an = action.squeeze(0).cpu().numpy()

        ctrl, delta = apply_ddelta(ctrl, delta, an)
        mjdata.ctrl[:] = ctrl
        for _ in range(SUBSTEPS):
            mujoco.mj_step(mjmodel, mjdata)

        if record_traj:
            traj_ctrl.append(ctrl.copy())
            traj_delta.append(delta.copy())

        vx   = float(mjdata.qvel[0])
        vx_err = max(0.0, abs(vx - TARGET_VX) - VX_TOL)
        tot_r += -VX_TRACK_W * vx_err ** 2
        done = is_done()
        if done:
            tot_r -= 5.0

        s = build_state(delta)

        if render and t % fe == 0:
            cam.lookat[0] = mjdata.qpos[0]
            cam.lookat[1] = mjdata.qpos[1]
            cam.azimuth   = (cam.azimuth + 0.2) % 360
            renderer.update_scene(mjdata, cam)
            frames.append(renderer.render().copy())

        if done:
            break

    dist = mjdata.qpos[0] - x0
    if renderer:
        renderer.close()
    if record_traj:
        return tot_r, dist, frames, np.array(traj_ctrl, dtype=np.float32), np.array(traj_delta, dtype=np.float32)
    if render:
        return tot_r, dist, frames
    return tot_r, dist


# ══════════════════════════════════════════════════════════════
#  Trajectory export
# ══════════════════════════════════════════════════════════════
JOINT_NAMES = [
    "FL_hip", "FL_knee", "FR_hip", "FR_knee",
    "RL_hip", "RL_knee", "RR_hip", "RR_knee",
]


def export_trajectory(traj: np.ndarray, tag: str = "best", delta_traj: np.ndarray = None):
    T    = traj.shape[0]
    base = f"trajectories/{tag}"

    np.save(f"{base}.npy", traj)
    print(f"  Saved {base}.npy  shape={traj.shape}")

    if delta_traj is not None:
        np.save(f"{base}_delta.npy", delta_traj)
        print(f"  Saved {base}_delta.npy  (delta buffer evolution)")

    header = "step_ms," + ",".join(JOINT_NAMES)
    rows = [f"{i*CTRL_DT*1000:.2f}," + ",".join(f"{v:.6f}" for v in row)
            for i, row in enumerate(traj)]
    with open(f"{base}.csv", "w") as f:
        f.write(header + "\n" + "\n".join(rows) + "\n")
    print(f"  Saved {base}.csv  ({T} rows)")

    scale = 10_000
    # Store as offset from neutral pose (OFFSETS) so the ESP32 only needs
    # to add the relative delta to CENTER — no absolute angle arithmetic.
    #   rel[step][j] = traj[step][j] - OFFSETS[j]
    #   on ESP32:  pulse = CENTER + rel * PULSE_PER_RAD * JOINT_DIR
    traj_rel = traj - OFFSETS[np.newaxis, :]        # shape (T, 8), centred at 0
    int16 = np.clip(np.round(traj_rel * scale), -32768, 32767).astype(np.int16)

    # Sanity: at step 0 (settled neutral) all values should be near zero
    max_abs = np.abs(traj_rel[0]).max()
    if max_abs > 0.05:
        print(f"  Warning: step-0 max |offset| = {max_abs:.4f} rad "
              f"(robot may not start at neutral pose)")

    lines = [
        f"/* Auto-generated trajectory  tag={tag}",
        f" * Steps: {T}  dt: {CTRL_DT*1000:.2f} ms  ({1/CTRL_DT:.0f} Hz)",
        f" * Joints: {', '.join(JOINT_NAMES)}",
        f" * Control scheme: delta-delta (ΔΔθ) — acceleration-level actions",
        f" * VALUES ARE RELATIVE TO NEUTRAL POSE (OFFSETS subtracted):",
        f" * neutral = [{', '.join(f'{v:.4f}' for v in OFFSETS)}] rad",
        f" * angle_rad  = OFFSET[j] + traj[step][j] / {scale}.0f",
        f" * pulse      = CENTER    + traj[step][j] / {scale}.0f * PULSE_PER_RAD * DIR[ch]",
        " */",
        "#pragma once", "#include <stdint.h>", "",
        f"#define TRAJ_STEPS  {T}",
        f"#define TRAJ_JOINTS 8",
        f"#define TRAJ_DT_MS  {int(round(CTRL_DT * 1000))}",
        f"#define TRAJ_SCALE  {scale}",
        f"/* neutral offsets (rad×TRAJ_SCALE) for reference:",
        f"   HIP_INIT  = {int(round(HIP_INIT  * scale))}",
        f"   KNEE_INIT = {int(round(KNEE_INIT * scale))} */", "",
        f"static const int16_t traj[{T}][8] = {{",
    ]
    for i, row in enumerate(int16):
        vals  = ", ".join(f"{int(v):6d}" for v in row)
        comma = "," if i < T - 1 else " "
        lines.append(f"  {{ {vals} }}{comma}  /* t={i*CTRL_DT*1000:7.1f} ms */")
    lines.append("};")
    with open(f"{base}.h", "w") as f:
        f.write("\n".join(lines) + "\n")
    print(f"  Saved {base}.h  ({T*8*2/1024:.1f} KB)")

    t_axis = np.arange(T) * CTRL_DT
    ncols  = 3 if delta_traj is not None else 2
    fig, axes = plt.subplots(4, ncols, figsize=(6*ncols, 10), sharex=True)

    for idx, name in enumerate(JOINT_NAMES):
        ax_pos = axes[idx // 2, (idx % 2) * (1 if ncols == 2 else 1)]
        ax_pos.plot(t_axis, traj[:, idx], lw=1.2)
        ax_pos.set_title(f"{name} pos", fontsize=9)
        ax_pos.set_ylabel("rad", fontsize=7)
        ax_pos.grid(True, alpha=0.3)

    if delta_traj is not None:
        for idx, name in enumerate(JOINT_NAMES):
            ax_d = axes[idx // 2, 2]
            ax_d.plot(t_axis, delta_traj[:, idx], lw=1.0, alpha=0.7, label=name)
        for r in range(4):
            axes[r, 2].legend(fontsize=6)
            axes[r, 2].set_title("delta buffer", fontsize=9)
            axes[r, 2].grid(True, alpha=0.3)

    for c in range(ncols):
        axes[-1, c].set_xlabel("time (s)")
    plt.suptitle(f"Trajectory (ΔΔθ) — {T} steps @ {1/CTRL_DT:.0f} Hz", fontsize=11)
    plt.tight_layout()
    plt.savefig(f"{base}_plot.png", dpi=120)
    plt.close()
    print(f"  Saved {base}_plot.png")


# ══════════════════════════════════════════════════════════════
#  ESP32 policy export
#  Writes a self-contained C header with:
#    • all actor weights/biases as float arrays
#    • all control constants (limits, deltas, offsets)
#    • a complete policy_step() function ready to #include
# ══════════════════════════════════════════════════════════════
def export_policy_header(path: str = "models/policy.h"):
    """Dump the current actor weights + a full C inference routine.

    On the ESP32 call once per control tick:
        policy_step(state, ctrl, delta);   // updates ctrl[] and delta[] in-place
    """
    sd = ac.state_dict()

    # actor weights  (pi.0 = first Linear, pi.2 = second Linear)
    W1 = sd["pi.0.weight"].cpu().numpy()   # (HIDDEN_DIM, STATE_DIM)
    b1 = sd["pi.0.bias"  ].cpu().numpy()   # (HIDDEN_DIM,)
    W2 = sd["pi.2.weight"].cpu().numpy()   # (ACTION_DIM, HIDDEN_DIM)
    b2 = sd["pi.2.bias"  ].cpu().numpy()   # (ACTION_DIM,)

    H = W1.shape[0]   # HIDDEN_DIM
    S = W1.shape[1]   # STATE_DIM
    A = W2.shape[0]   # ACTION_DIM

    def arr1d(name, vec):
        vals = ", ".join(f"{v:.8f}f" for v in vec.flatten())
        return f"static const float {name}[{vec.size}] = {{{vals}}};\n"

    def arr2d(name, mat, comment=""):
        rows, cols = mat.shape
        out = [f"/* {comment} */" if comment else "",
               f"static const float {name}[{rows}][{cols}] = {{"]
        for i, row in enumerate(mat):
            vals  = ", ".join(f"{v:.8f}f" for v in row)
            comma = "," if i < rows - 1 else ""
            out.append(f"  {{ {vals} }}{comma}")
        out.append("};\n")
        return "\n".join(out) + "\n"

    def flt(name, val, comment=""):
        c = f"  /* {comment} */" if comment else ""
        return f"static const float {name} = {val:.8f}f;{c}\n"

    lines = []
    lines.append(f"""\
#pragma once
/*
 * Auto-generated policy header — DO NOT EDIT BY HAND
 *
 * Architecture : {S}-input  ->  ReLU({H})  ->  tanh({A})-output
 * Training     : PPO  delta-delta (acceleration-level) control
 * Target speed : {TARGET_VX:.4f} m/s  ({1.0/TARGET_VX:.1f} s per metre)
 * Control dt   : {CTRL_DT*1000:.2f} ms  ({1/CTRL_DT:.0f} Hz)
 *
 * Flash footprint:
 *   W1 {H}x{S} = {H*S*4} bytes
 *   W2 {A}x{H} = {A*H*4} bytes
 *   total approx {(H*S + H + A*H + A)*4} bytes
 *
 * Quick-start (Arduino/ESP-IDF):
 *
 *   #include "policy.h"
 *
 *   float ctrl[POLICY_ACTION_DIM]  = {{HIP0, KNEE0, HIP0, KNEE0,
 *                                      HIP0, KNEE0, HIP0, KNEE0}};
 *   float delta[POLICY_ACTION_DIM] = {{0}};
 *
 *   void loop() {{
 *     float state[POLICY_STATE_DIM];
 *     read_encoders(state);          // jpos[0..7]
 *     read_imu_jvel(state + 8);      // jvel[0..7]
 *     memcpy(state + 16, delta, sizeof(delta));  // current delta buffer
 *     policy_step(state, ctrl, delta);
 *     send_joint_targets(ctrl);
 *     delay(POLICY_CTRL_DT_MS);
 *   }}
 */

#include <math.h>
#include <string.h>

#define POLICY_STATE_DIM  {S}
#define POLICY_HIDDEN_DIM {H}
#define POLICY_ACTION_DIM {A}
#define POLICY_CTRL_DT_MS {int(round(CTRL_DT * 1000))}

#define POLICY_HIP_INIT   {HIP_INIT:.6f}f
#define POLICY_KNEE_INIT  {KNEE_INIT:.6f}f

""")

    lines.append("/* ---- Layer 1  W1[HIDDEN][STATE] ---- */\n")
    lines.append(arr2d("POLICY_W1", W1, f"W1[{H}][{S}]: weights for hidden layer"))
    lines.append("/* ---- Layer 1  b1[HIDDEN] ---- */\n")
    lines.append(arr1d("POLICY_b1", b1))
    lines.append("\n/* ---- Layer 2  W2[ACTION][HIDDEN] ---- */\n")
    lines.append(arr2d("POLICY_W2", W2, f"W2[{A}][{H}]: weights for output layer"))
    lines.append("/* ---- Layer 2  b2[ACTION] ---- */\n")
    lines.append(arr1d("POLICY_b2", b2))

    lines.append("\n/* ---- Control constants ---- */\n")
    lines.append(flt("POLICY_MAX_DDELTA",  MAX_DDELTA,  "acceleration scale (rad/step)"))
    lines.append(flt("POLICY_DELTA_LIMIT", DELTA_LIMIT, "max |delta| (rad/step)"))
    lines.append(flt("POLICY_TARGET_VX",   TARGET_VX,   "desired forward speed (m/s)"))

    lines.append(f"\nstatic const float POLICY_JOINT_LO[{A}] = {{"
                 + ", ".join(f"{v:.6f}f" for v in JOINT_LO) + "};\n")
    lines.append(f"static const float POLICY_JOINT_HI[{A}] = {{"
                 + ", ".join(f"{v:.6f}f" for v in JOINT_HI) + "};\n")
    lines.append(f"static const float POLICY_OFFSETS[{A}]   = {{"
                 + ", ".join(f"{v:.6f}f" for v in OFFSETS)   + "};\n")

    lines.append(f"""
/* ---- policy_step: run one control tick -------------------------
 *
 * state[{S}] : input  -- {{ jpos[8], jvel[8], delta[8] }}
 *   jpos  = joint angles (rad) from encoders, order: FL_hip, FL_knee,
 *           FR_hip, FR_knee, RL_hip, RL_knee, RR_hip, RR_knee
 *   jvel  = joint velocities (rad/s) from encoders / finite difference
 *   delta = persistent delta buffer — zero-initialise at boot, then pass
 *           the same array every tick so it carries state between calls
 *
 * ctrl[{A}]  : in/out -- joint position targets (rad); send to servos after call
 * delta[{A}] : in/out -- persistent buffer, updated in-place; DO NOT reset between ticks
 *
 * Computation:
 *   h[i]      = ReLU( sum_j W1[i][j]*state[j] + b1[i] )
 *   mean[k]   = sum_i W2[k][i]*h[i] + b2[k]
 *   action[k] = tanh(mean[k])
 *   delta[k]  = clamp(delta[k] + action[k]*MAX_DDELTA, -DELTA_LIMIT, +DELTA_LIMIT)
 *   ctrl[k]   = clamp(ctrl[k]  + delta[k],              JOINT_LO[k],  JOINT_HI[k])
 */
static inline void policy_step(
    const float state[POLICY_STATE_DIM],
    float       ctrl [POLICY_ACTION_DIM],
    float       delta[POLICY_ACTION_DIM])
{{
    int i, j, k;

    /* --- hidden layer: h = ReLU(W1 @ state + b1) ------------ */
    float h[POLICY_HIDDEN_DIM];
    for (i = 0; i < POLICY_HIDDEN_DIM; i++) {{
        float acc = POLICY_b1[i];
        for (j = 0; j < POLICY_STATE_DIM; j++)
            acc += POLICY_W1[i][j] * state[j];
        h[i] = (acc > 0.0f) ? acc : 0.0f;
    }}

    /* --- output layer: mean = W2 @ h + b2, action = tanh ---- */
    float action[POLICY_ACTION_DIM];
    for (k = 0; k < POLICY_ACTION_DIM; k++) {{
        float acc = POLICY_b2[k];
        for (i = 0; i < POLICY_HIDDEN_DIM; i++)
            acc += POLICY_W2[k][i] * h[i];
        action[k] = tanhf(acc);
    }}

    /* --- two-level integration ------------------------------- */
    for (k = 0; k < POLICY_ACTION_DIM; k++) {{
        /* acceleration -> velocity */
        delta[k] += action[k] * POLICY_MAX_DDELTA;
        if      (delta[k] >  POLICY_DELTA_LIMIT) delta[k] =  POLICY_DELTA_LIMIT;
        else if (delta[k] < -POLICY_DELTA_LIMIT) delta[k] = -POLICY_DELTA_LIMIT;

        /* velocity -> position */
        ctrl[k] += delta[k];
        if      (ctrl[k] > POLICY_JOINT_HI[k]) ctrl[k] = POLICY_JOINT_HI[k];
        else if (ctrl[k] < POLICY_JOINT_LO[k]) ctrl[k] = POLICY_JOINT_LO[k];
    }}
}}
""")

    with open(path, "w") as f:
        f.writelines(lines)

    total_params = H*S + H + A*H + A
    print(f"  Saved {path}")
    print(f"  Actor: W1({H}x{S}) + b1({H}) + W2({A}x{H}) + b2({A}) "
          f"= {total_params} floats = {total_params*4/1024:.1f} KB")


# ══════════════════════════════════════════════════════════════
#  Eval-only mode
# ══════════════════════════════════════════════════════════════
if EVAL_ONLY:
    assert LOAD_MODEL
    print("Eval mode...")
    _, bd, frames, traj, delta_traj = rollout_eval(render=True, record_traj=True)
    if frames:
        media.write_video("movies/nn_ppo_ddtheta_eval.mp4", frames, fps=60)
        print(f"Saved: movies/nn_ppo_ddtheta_eval.mp4  dist={bd:.3f}m")
    export_trajectory(traj, tag="eval", delta_traj=delta_traj)
    export_policy_header("models/policy_eval.h")
    import sys; sys.exit(0)


# ══════════════════════════════════════════════════════════════
#  Training loop
# ══════════════════════════════════════════════════════════════
best_dist = best_dist_loaded
best_w    = None
nm        = 0
ppo_update_count = 0

if LOAD_MODEL and best_dist_loaded > -np.inf:
    best_w = {k: v.clone() for k, v in ac.state_dict().items()}

START_EP = ep_loaded + 1 if LOAD_MODEL else 0
END_EP   = START_EP + N_EP

print(f"\nNN+PPO (ΔΔθ) | ep {START_EP}→{END_EP} | state={STATE_DIM} act={ACTION_DIM} "
      f"| MAX_DDELTA={MAX_DDELTA} DELTA_LIMIT={DELTA_LIMIT} rad "
      f"| align_w={ALIGN_W} yaw_w={YAW_W} "
      f"| buffer={N_STEPS_PER_UPDATE} | epochs={PPO_EPOCHS}\n")

for ep in trange(START_EP, END_EP):
    r, d, last_val = rollout_train()
    rl.append(r); dl.append(d)

    if d > best_dist:
        best_dist = d
        best_w = {k: v.clone() for k, v in ac.state_dict().items()}

    cl = al = 0.0
    if len(ppo_buf) >= N_STEPS_PER_UPDATE:
        cl, al = update_ppo(last_val)
        ppo_update_count += 1
    cll.append(cl); all_.append(al)

    if ep % 10 == 0:
        log_std_val = ac.log_std.data.mean().item()
        print(f"  ep {ep:5d} | dist={d:.3f}m avg10={np.mean(dl[-10:]):.3f} | "
              f"R={r:.2f} | buf={len(ppo_buf)} | "
              f"σ={math.exp(log_std_val):.3f} | "
              f"VL={cl:.4f} PL={al:.4f} | updates={ppo_update_count}")

    if ep % RENDER_EVERY == 0 and ep > START_EP and best_w:
        train_w = {k: v.clone() for k, v in ac.state_dict().items()}
        ac.load_state_dict(best_w)
        _, bd, frames = rollout_eval(render=True)
        if frames:
            out = f"movies/nn_ppo_ddtheta_{nm:03d}_ep{ep:04d}_{bd:.2f}m.mp4"
            media.write_video(out, frames, fps=60)
            print(f"  → {out}")
        nm += 1
        ac.load_state_dict(train_w)

    if ep % 50 == 49:
        torch.save({
            "ac": ac.state_dict(), "opt": opt.state_dict(),
            "best_w": best_w,
            "best_dist": best_dist, "ep": ep,
            "rl": rl, "dl": dl, "cll": cll, "all_": all_,
        }, f"models/nn_ppo_ddtheta_ep{ep}.pth")

        w  = 20
        sm = lambda x: np.convolve(x, np.ones(w)/w, "valid")
        if len(dl) >= w:
            fig, ax = plt.subplots(2, 2, figsize=(12, 8))
            ax[0, 0].plot(sm(dl),  color="#2ecc71"); ax[0, 0].set_title("Distance (m)")
            ax[0, 1].plot(sm(rl),  color="#3498db"); ax[0, 1].set_title("Reward")
            ax[1, 0].plot(sm(cll), color="#e74c3c"); ax[1, 0].set_title("Value Loss")
            ax[1, 1].plot(sm(all_),color="#f39c12"); ax[1, 1].set_title("Policy Loss")
            for a_ in ax.flat:
                a_.grid(True, alpha=0.3); a_.set_xlabel("ep")
            plt.suptitle(f"NN+PPO (ΔΔθ)  ep {ep} | best={best_dist:.3f}m")
            plt.tight_layout()
            plt.savefig(f"plots/nn_ppo_ddtheta_ep{ep}.png", dpi=120)
            plt.close()


CTRL_DT = 10.0 ms  (100 Hz)
State dim = 24  |  Action dim = 8
MAX_DDELTA = 0.02 rad/step  |  DELTA_LIMIT = 0.05 rad/step
Device: cpu

NN+PPO (ΔΔθ) | ep 0→10000 | state=24 act=8 | MAX_DDELTA=0.02 DELTA_LIMIT=0.05 rad | align_w=0.0 yaw_w=0.2 | buffer=4096 | epochs=10



  0%|          | 1/10000 [00:00<44:39,  3.73it/s]

  ep     0 | dist=0.098m avg10=0.098 | R=-46.77 | buf=800 | σ=1.649 | VL=0.0000 PL=0.0000 | updates=0


  0%|          | 11/10000 [00:02<38:50,  4.29it/s]

  ep    10 | dist=0.253m avg10=0.136 | R=4.35 | buf=3339 | σ=1.609 | VL=0.0000 PL=0.0000 | updates=1


...

100%|█████████▉| 9982/10000 [9:42:22<00:03,  5.15it/s]

  ep  9980 | dist=1.854m avg10=1.909 | R=340.63 | buf=3200 | σ=0.491 | VL=0.0000 PL=0.0000 | updates=1660


100%|█████████▉| 9992/10000 [9:42:25<00:01,  4.90it/s]

  ep  9990 | dist=1.857m avg10=1.901 | R=338.83 | buf=1600 | σ=0.490 | VL=0.0000 PL=0.0000 | updates=1662


100%|██████████| 10000/10000 [9:42:27<00:00,  3.49s/it]


In [2]:


# ══════════════════════════════════════════════════════════════
#  Final render + trajectory export
# ══════════════════════════════════════════════════════════════
print(f"\n=== Best dist: {best_dist:.3f}m ===")

if best_w:
    ac.load_state_dict(best_w)
    torch.save({"ac": best_w, "best_dist": best_dist},
               "models/nn_ppo_ddtheta_best.pth")

    print("\nRecording final video + exporting trajectory...")
    _, bd, frames, traj, delta_traj = rollout_eval(render=True, record_traj=True)
    if frames:
        media.write_video("movies/nn_ppo_ddtheta_final.mp4", frames, fps=60)
        print(f"Final: movies/nn_ppo_ddtheta_final.mp4  dist={bd:.3f}m")
    export_trajectory(traj, tag="best", delta_traj=delta_traj)
    export_policy_header("models/policy.h")

    # ── One-cycle detection ───────────────────────────────────
    one_cycle_found = False
    T = traj.shape[0]
    if T > 64:
        sig   = traj[:, 0] - traj[:, 0].mean()
        fft   = np.fft.rfft(sig)
        freqs = np.fft.rfftfreq(T, d=CTRL_DT)
        dom_f = freqs[np.argmax(np.abs(fft[1:])) + 1]
        if dom_f > 0.1:
            spc = int(round(1.0 / (dom_f * CTRL_DT)))
            if T >= 2 * spc:
                export_trajectory(traj[:spc], tag="one_cycle", delta_traj=delta_traj[:spc])
                print(f"\nOne-cycle: {spc} steps ({1/dom_f:.2f}s, {dom_f:.2f} Hz)")
                one_cycle_found = True

    # ══════════════════════════════════════════════════════════
    #  ESP32 export bundle
    #  Collects policy.h + the best available trajectory.h
    #  into export/ so you can drop both files into the Arduino
    #  sketch folder and compile immediately.
    #
    #  Preference: one_cycle.h (shorter, loops cleanly on robot)
    #              → falls back to best.h if no cycle detected
    # ══════════════════════════════════════════════════════════
    import shutil
    os.makedirs("export", exist_ok=True)

    # policy.h — actor weights + policy_step() C function
    shutil.copy("models/policy.h", "export/policy.h")
    print("\nESP32 export bundle → export/")
    print("  Copied: models/policy.h  →  export/policy.h")

    # trajectory.h — pick one_cycle if available, otherwise best
    if one_cycle_found:
        src_traj = "trajectories/one_cycle.h"
        label    = "one_cycle"
    else:
        src_traj = "trajectories/best.h"
        label    = "best"

    shutil.copy(src_traj, "export/trajectory.h")
    print(f"  Copied: {src_traj}  →  export/trajectory.h  ({label})")

    print("\n  Sketch folder contents should be:")
    print("    albert_trajectory/")
    print("      albert_trajectory.ino")
    print("      trajectory.h          ← from export/trajectory.h")
    print("    albert_firmware/")
    print("      albert_firmware.ino")
    print("      policy.h              ← from export/policy.h")


=== Best dist: 2.114m ===

Recording final video + exporting trajectory...
Final: movies/nn_ppo_ddtheta_final.mp4  dist=2.063m
  Saved trajectories/best.npy  shape=(800, 8)
  Saved trajectories/best_delta.npy  (delta buffer evolution)
  Saved trajectories/best.csv  (800 rows)
  Saved trajectories/best.h  (12.5 KB)
  Saved trajectories/best_plot.png
  Saved models/policy.h
  Actor: W1(64x24) + b1(64) + W2(8x64) + b2(8) = 2120 floats = 8.3 KB
  Saved trajectories/one_cycle.npy  shape=(25, 8)
  Saved trajectories/one_cycle_delta.npy  (delta buffer evolution)
  Saved trajectories/one_cycle.csv  (25 rows)
  Saved trajectories/one_cycle.h  (0.4 KB)
  Saved trajectories/one_cycle_plot.png

One-cycle: 25 steps (0.25s, 4.00 Hz)

ESP32 export bundle → export/
  Copied: models/policy.h  →  export/policy.h
  Copied: trajectories/one_cycle.h  →  export/trajectory.h  (one_cycle)

  Sketch folder contents should be:
    albert_trajectory/
      albert_trajectory.ino
      trajectory.h          ← fro